# Qiskit으로 원하는 state 만들기
**Qiskit Fall Fest 2026 @ SNU · 튜토리얼**

회로를 만들고 결과를 확인하면서 Qiskit을 익혀 봅시다. 중간에 **튜토리얼 미션 1·2**가 있습니다.
각 미션을 풀고 출력되는 **통과 화면과 확인 코드**를 현장의 동아리 운영위원회에 보여 주세요.
운영위원회 확인 후 키링 등 기념품을 받을 수 있습니다. 지급 방법은 현장 안내를 따라 주세요.

**메인 해커톤**은 별도의 [`problem.md`](https://github.com/SNUQuantums/snu-qff2026-problem/blob/main/problem.md)에 있습니다.
튜토리얼 미션 완료 여부와 관계없이 참여할 수 있습니다.

ChatGPT, Claude, GitHub Copilot 등 **AI 도구를 자유롭게 사용해도 됩니다.** 개념 학습, 코드 작성·수정,
디버깅, 회로 설계 등 모든 과정에서 사용할 수 있습니다.

| 순서 | 해 볼 일 |
|---|---|
| 준비 | 설치와 실행 환경 |
| Part 1 | gate → Bell → GHZ 준비 · 미션 1 |
| Part 2 | 격자와 noise → 회로 개선 · 미션 2 |
| Part 3 · 선택 심화 | state를 보존하는 검사 → 실행 선택 → 보정 기록 |
| Part 4 | 자동 state 준비 → 메인 해커톤 |

Zoom 설명 중에는 진행자가 안내하는 곳까지 실행하면 됩니다. 설명을 놓쳤다면 위에서부터 읽고 실행해 보세요.

## 0. 준비

**Google Colab**에서는 배포받은 `tutorial.ipynb`를 업로드해 열면 됩니다. 로컬에서는 Jupyter나 VS Code로 열고 Python 커널을 선택하세요.
아래 설치 셀부터 실행합니다. 이 튜토리얼은 **Python 3.11 이상**을 기준으로 합니다.

셀 왼쪽의 ▶ 또는 `Shift + Enter`로 실행할 수 있습니다. 앞에서 정의한 변수를 뒤에서 사용하므로 순서대로 실행하세요.
`NameError`가 나면 import 셀부터 다시 실행해 보세요.

튜토리얼 미션에는 IBM 계정, 실제 양자컴퓨터, 메인 해커톤 채점기가 필요하지 않습니다.

In [ ]:
%pip install -q "qiskit==2.5.2" matplotlib pylatexenc sympy

In [ ]:
import hashlib
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from IPython.display import display
from qiskit import QuantumCircuit, transpile, qasm2
from qiskit.quantum_info import Statevector, DensityMatrix, Kraus, state_fidelity
from qiskit.transpiler import CouplingMap
from qiskit.visualization import plot_histogram

print("Qiskit", qiskit.__version__)
print("준비 완료!")

## Part 1. 원하는 state 만들기

### 1.1 Qubit 하나의 state 바꾸기

Qubit 하나의 state는 $a|0\rangle+b|1\rangle$로 적습니다. $a,b$는 **amplitude**이며 복소수일 수 있습니다.
측정하면 0 또는 1이 나오고, 그 확률은 각각 $|a|^2$, $|b|^2$입니다.

`QuantumCircuit(1)`은 qubit 하나의 회로를 만듭니다. 여기서는 처음 state를 $|0\rangle$으로 둡니다.

| 동작 | Qiskit 코드 | 예 |
|---|---|---|
| X gate | `qc.x(0)` | $|0\rangle\to|1\rangle$ |
| H gate | `qc.h(0)` | $|0\rangle\to(|0\rangle+|1\rangle)/\sqrt2$ |
| 회로 그리기 | `qc.draw("mpl")` | gate를 붙인 순서 확인 |
| state 계산 | `Statevector.from_instruction(qc)` | noise 없는 회로의 출력 확인 |

먼저 X gate를 실행해 보세요. 그다음 `qc.x(0)`을 `qc.h(0)`으로 바꾸고 다시 실행해 보세요.

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)  # h(0)으로 바꾸면 어떻게 될까요?

display(qc.draw("mpl"))
state = Statevector.from_instruction(qc)
display(state.draw("latex"))
print("측정 확률:", state.probabilities_dict())

### Qiskit의 bit 순서

Qubit 번호는 0부터 셉니다. Ket 문자열에서는 **qubit 0이 오른쪽 끝**에 있습니다.
이 노트북과 메인 문제는 모두 이 순서를 사용합니다.

예를 들어 $|q_2q_1q_0\rangle=|001\rangle$은 qubit 0만 1인 state입니다.

In [ ]:
bits = QuantumCircuit(3)
bits.x(0)
print(Statevector.from_instruction(bits).probabilities_dict())  # {'001': 1.0}

### 1.2 두 qubit 연결하기

CNOT은 control qubit이 1인 항에서 target qubit을 뒤집습니다.
`qc.cx(control, target)`처럼 씁니다.

아래 회로에서는 먼저 H로 $|00\rangle$과 $|01\rangle$을 겹치고, CNOT으로 $|01\rangle$을 $|11\rangle$로 바꿉니다.

$$|00\rangle\;\longrightarrow\;\frac{|00\rangle+|01\rangle}{\sqrt2}
\;\longrightarrow\;\frac{|00\rangle+|11\rangle}{\sqrt2}.$$

마지막 state는 **Bell state**입니다. 두 qubit을 측정하면 00 또는 11이 나옵니다.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

display(bell.draw("mpl"))
bell_state = Statevector.from_instruction(bell)
display(bell_state.draw("latex"))
display(plot_histogram(bell_state.probabilities_dict(), title="Exact probabilities"))

### 확률과 실제 측정 횟수

`probabilities_dict()`는 계산된 확률을 보여 줍니다. `sample_counts(shots)`는 그 확률에 따라 측정 결과를 뽑습니다.
`shots`는 반복 횟수입니다. 아래 값을 32, 256, 4096으로 바꿔 보세요.

실제 회로의 측정은 `measure`로 적습니다. 아래에서는 측정 회로를 그려 보고, 결과 표본은 앞에서 계산한 state에서 얻습니다.
`Statevector`로 회로를 계산할 때는 측정 전 회로를 사용합니다.

In [ ]:
measured = bell.copy()
measured.measure_all()
display(measured.draw("mpl"))

bell_state.seed(2026)
counts = bell_state.sample_counts(shots=256)
print(counts)
display(plot_histogram(counts, title="256 samples"))

### 1.3 튜토리얼 미션 1 — 세 qubit을 연결하기

다음 state를 만드는 회로를 작성하세요.

$$|\psi_1\rangle=\frac{|000\rangle+|111\rangle}{\sqrt2}.$$

- Qubit은 3개이며 처음에는 모두 0입니다.
- **H와 CNOT**을 사용하세요. 측정은 넣지 않습니다.
- Bell state를 만든 방법을 qubit 하나에 더 적용해 보세요.

아래 함수의 `TODO` 부분을 채우고 실행하세요. 그다음 확인 셀을 실행합니다.

In [ ]:
def mission1_circuit():
    qc = QuantumCircuit(3)
    # TODO: H와 CNOT으로 목표 state를 만드세요.
    return qc

m1 = mission1_circuit()
display(m1.draw("mpl"))

### 미션 확인하기

두 state가 얼마나 같은지 **fidelity**로 확인합니다. 순수한 state끼리는
$F=|\langle\psi_{\mathrm{target}}|\psi_{\mathrm{out}}\rangle|^2$이며, 같으면 1입니다.
전체 state에 공통으로 붙는 global phase는 통과 여부에 영향을 주지 않습니다.
아래 도구는 미션 1용입니다. 그대로 실행하세요.


In [ ]:
def ghz_target(n):
    target = np.zeros(2**n, dtype=complex)
    target[0] = target[-1] = 1 / np.sqrt(2)
    return Statevector(target)


def mission_token(circuit, mission, participant):
    payload = f"SNU-QFF2026|tutorial-v2|{mission}|{participant}|{qasm2.dumps(circuit)}"
    digest = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:10].upper()
    token = f"QFF26-M{mission}-{digest}"
    print(f"\n튜토리얼 미션 {mission} 통과!")
    print(f"참가자: {participant}\n확인 코드: {token}")
    print("이 통과 화면을 현장의 동아리 운영위원회에 보여 주세요.")
    return token


def check_mission(circuit, mission, participant):
    if mission != 1:
        print("미션 2는 아래의 check_mission2를 사용하세요.")
        return None
    participant = participant.strip()
    if not participant or participant == "여기에 이름 또는 닉네임":
        print("PARTICIPANT에 이름 또는 닉네임을 입력해 주세요.")
        return None
    if not isinstance(circuit, QuantumCircuit) or circuit.num_qubits != 3:
        print("QuantumCircuit(3)으로 만든 회로를 넣어 주세요.")
        return None
    allowed = {"h", "cx", "barrier"}
    if any(inst.operation.name not in allowed for inst in circuit.data):
        print("미션 1에서는 h, cx, barrier만 사용하세요.")
        return None
    fidelity = float(state_fidelity(Statevector.from_instruction(circuit), ghz_target(3)))
    print(f"목표 state와의 fidelity: {fidelity:.10f}")
    if not np.isclose(fidelity, 1.0, atol=1e-10, rtol=0):
        print("아직 목표 state와 다릅니다. 회로를 수정하고 다시 실행하세요.")
        return None
    return mission_token(circuit, 1, participant)

In [ ]:
PARTICIPANT = "여기에 이름 또는 닉네임"
check_mission(m1, mission=1, participant=PARTICIPANT)

### 1.4 확률이 같으면 같은 state일까요?

다음 두 state는 0과 1이 나올 확률이 각각 1/2로 같습니다.

$$|+\rangle=\frac{|0\rangle+|1\rangle}{\sqrt2},\qquad
|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}.$$

하지만 $|1\rangle$ 항의 부호, 즉 **상대위상**이 다릅니다. Z gate는 이 부호를 바꿉니다.
두 state에 H를 한 번 더 적용하면 각각 $|0\rangle$, $|1\rangle$이 되어 차이를 측정할 수 있습니다.

In [ ]:
plus = QuantumCircuit(1)
plus.h(0)
minus = plus.copy()
minus.z(0)

for label, circuit in [("plus", plus), ("minus", minus)]:
    sv = Statevector.from_instruction(circuit)
    print(label, "측정 확률:", sv.probabilities_dict())
    display(sv.draw("latex"))
    after_h = circuit.copy()
    after_h.h(0)
    print("H를 한 번 더 적용:", Statevector.from_instruction(after_h).probabilities_dict())

### Z, S, T로 상대위상 조절하기

세 gate는 $|0\rangle$을 그대로 두고 $|1\rangle$의 amplitude에 다음 값을 곱합니다.

| Gate | 곱하는 값 | Qiskit |
|---|---|---|
| Z | $-1=e^{i\pi}$ | `qc.z(q)` |
| S | $i=e^{i\pi/2}$ | `qc.s(q)` |
| T | $e^{i\pi/4}$ | `qc.t(q)` |

`sdg`, `tdg`는 각각 S와 T의 역연산입니다. 아래에서는 S를 적용한 state를 확인합니다.
`qc.s(0)`을 다른 gate로 바꾸어 amplitude를 비교해 보세요.

In [ ]:
phase_demo = QuantumCircuit(1)
phase_demo.h(0)
phase_demo.s(0)
display(phase_demo.draw("mpl"))
display(Statevector.from_instruction(phase_demo).draw("latex"))

## Part 2. Noise가 있는 회로를 시뮬레이션하고 개선하기

### 2.1 실제 연결에 맞춰 회로 바꾸기

지금까지는 어떤 qubit끼리든 CNOT을 걸었습니다. 메인 문제의 칩은 6×6 격자이며,
상하좌우로 이웃한 쌍에만 CNOT을 적용할 수 있습니다.
아래에서는 세 qubit이 일렬로 연결된 경우를 먼저 봅니다.
연결되지 않은 쌍의 연산을 구현하려면 state를 이동하는 등 추가 연산이 필요합니다.
SWAP은 CNOT **세 개**로 구현할 수 있습니다. CNOT 하나가 SWAP을 대신하는 것은 아닙니다.

In [ ]:
line = CouplingMap([[0, 1], [1, 0], [1, 2], [2, 1]])
source = QuantumCircuit(3)
source.h(0)
source.cx(0, 2)

routed_line = transpile(source, coupling_map=line, initial_layout=[0, 1, 2],
                        basis_gates=["h", "cx"],
                        optimization_level=0, seed_transpiler=2026)
display(source.draw("mpl"))
display(routed_line.draw("mpl"))
print("원래 gate 수:", dict(source.count_ops()))
print("변환 후 gate 수:", dict(routed_line.count_ops()))
print("원래 qubit 0, 1, 2의 최종 위치:", routed_line.layout.final_index_layout())

`final_index_layout()[i]`는 원래 qubit `i`의 state가 마지막에 놓인 physical qubit 번호입니다.
메인 해커톤에서는 이 정보를 `data_qubits`로 제출합니다.

`optimization_level`을 0에서 1, 2, 3으로 바꾸거나 `seed_transpiler`를 바꿔 보세요.
Gate 수와 배치가 달라질 수 있습니다. Gate 수가 줄어드는지는 직접 비교해 보세요.

### 2.2 Noise가 있는 4-qubit GHZ

미션 1에서는 gate가 정확하게 작동한다고 가정했습니다. 이제 gate를 적용하거나
qubit이 기다리는 동안 오류가 생기는 모델을 사용합니다.
목표는 익숙한 GHZ state를 qubit 네 개로 만드는 것입니다.

$$|\mathrm{GHZ}_4\rangle=\frac{|0000\rangle+|1111\rangle}{\sqrt2}.$$

칩 맨 윗줄의 **physical qubit 0, 1, 2, 3**을 사용합니다. 이웃한 번호끼리만 CNOT을 걸 수 있습니다.
Noise가 없을 때는 목표와 같아야 하고, noise가 있을 때는 fidelity가 높을수록 좋습니다.

### 메인 채점기의 noise와 그림 도구 준비

아래 셀은 레포를 준비하고 필요한 패키지를 설치합니다. 시간이 조금 걸릴 수 있습니다.
메인 문제와 같은 noise 삽입·스케줄 코드를 사용하지만, GHZ의 점수는 별도 도구로 계산합니다.
`tprep.check`는 7-qubit 메인 목표용이므로 여기서는 사용하지 않습니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path("snu-qff2026-problem")
if not Path("tprep/noise.py").exists():
    if not repo.exists():
        subprocess.run([
            "git", "clone", "https://github.com/SNUQuantums/snu-qff2026-problem.git", str(repo)
        ], check=True)
    sys.path.insert(0, str(repo.resolve()))
    requirements = repo / "requirements.txt"
else:
    requirements = Path("requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

import tprep
print("tprep", tprep.__version__)

### GHZ 시뮬레이터 — 아래 코드는 그대로 실행하세요

4-qubit state를 `DensityMatrix`로 계산합니다. 오류가 발생한 경우와 발생하지 않은 경우를
확률에 따라 합치므로 이 실습의 fidelity에는 shots에 따른 통계 오차가 없습니다.
메인 채점은 더 큰 회로를 처리하기 위해 다른 방법을 사용합니다.

이 실습은 측정·reset 없이 `h`, `cx`, `id`, `barrier`로 만든 회로를 대상으로 합니다.
초기화·gate·idle noise는 현재 메인 문제의 공식 모델을 따릅니다.

In [ ]:
from itertools import product
from qiskit.quantum_info import Pauli
from tprep.qasm import to_stim
from tprep.noise import apply_noise, schedule
from tprep.parse import parse

TUTORIAL_P = 0.01  # 차이를 관찰하기 위한 값. 메인 문제의 공식 p=0.001과 다릅니다.


def ghz_text(circuit):
    if circuit.num_qubits != 4 or circuit.num_clbits:
        raise ValueError("측정 없이 QuantumCircuit(4)로 만드세요.")
    allowed = {"h", "cx", "id", "barrier"}
    if any(i.operation.name not in allowed for i in circuit.data):
        raise ValueError("이 실습에서는 h, cx, id, barrier만 사용하세요.")
    text, errors = to_stim(tprep.submission_text(circuit))
    if errors:
        raise ValueError("; ".join(errors))
    return text


def depolarizing_channel(probability, n):
    labels = ["".join(x) for x in product("IXYZ", repeat=n)]
    return Kraus([
        np.sqrt(1 - probability) * np.eye(2**n),
        *[np.sqrt(probability / (4**n - 1)) * Pauli(s).to_matrix() for s in labels[1:]],
    ])


def noisy_ghz(circuit, p=TUTORIAL_P):
    text = ghz_text(circuit)
    rho = DensityMatrix.from_label("0000")
    for ins in parse(apply_noise(text, p)):
        if ins.name in {"DEPOLARIZE1", "DEPOLARIZE2"}:
            strength = float(ins.text.split("(", 1)[1].split(")", 1)[0])
            n = 1 if ins.name == "DEPOLARIZE1" else 2
            channel = depolarizing_channel(strength, n)
            for j in range(0, len(ins.qubits), n):
                rho = rho.evolve(channel, qargs=list(ins.qubits[j:j + n]))
        elif ins.name == "X_ERROR":
            strength = float(ins.text.split("(", 1)[1].split(")", 1)[0])
            channel = Kraus([np.sqrt(1 - strength) * np.eye(2),
                             np.sqrt(strength) * Pauli("X").to_matrix()])
            for q in ins.qubits:
                rho = rho.evolve(channel, qargs=[q])
        elif ins.name in {"H", "CX", "I"}:
            gate = QuantumCircuit(4)
            if ins.name == "CX":
                for j in range(0, len(ins.qubits), 2):
                    gate.cx(*ins.qubits[j:j + 2])
            else:
                for q in ins.qubits:
                    getattr(gate, "h" if ins.name == "H" else "id")(q)
            rho = rho.evolve(gate)
        else:
            raise ValueError(f"이 실습에서 지원하지 않는 연산: {ins.name}")
    return rho


def ghz_report(circuit, p=TUTORIAL_P):
    text = ghz_text(circuit)
    _, steps, gaps = schedule(text)
    depth = 1 + max((step for groups in steps for _, step in groups), default=-1)
    ideal = float(state_fidelity(Statevector.from_instruction(circuit), ghz_target(4)))
    noisy = float(state_fidelity(noisy_ghz(circuit, p), ghz_target(4)))
    print(f"noise 없음 F = {ideal:.10f}")
    print(f"noise 있음 F = {noisy:.6f}, 1-F = {1-noisy:.6f} (p={p})")
    print(f"시간 step = {depth}, idle qubit-step = {sum(n for _, _, _, n in gaps)}")
    print("gate 수:", dict(circuit.count_ops()))
    return noisy


def show_ghz(circuit):
    # 짧은 회로에서도 그림과 범례가 잘 보이도록 크기를 조정합니다.
    import textwrap
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Tight layout not applied.*")
        fig = tprep.draw(circuit, data_qubits=[0, 1, 2, 3])
    fig.set_size_inches(11, 5)
    fig.axes[0].get_subplotspec().get_gridspec().set_width_ratios([1, 1.2])
    for label in fig.axes[1].texts:
        if label.get_text().startswith("small red dot"):
            legend = textwrap.fill(label.get_text(), width=60)
            label.set_text("")
            fig.text(0.52, 0.04, legend, fontsize=8, va="bottom")
    fig.tight_layout(rect=(0, 0.15, 1, 1))
    display(fig)
    return fig


In [ ]:
def starter_ghz():
    qc = QuantumCircuit(4)
    qc.h(0)
    qc.h(0)  # 처음 두 H를 지우면 어떻게 될까요?
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.cx(2, 3)
    return qc


starter = starter_ghz()
show_ghz(starter)
starter_fidelity = ghz_report(starter)
display(plot_histogram(
    [ghz_target(4).probabilities_dict(), noisy_ghz(starter).probabilities_dict()],
    legend=["Ideal GHZ", "Noisy output"], title="Z-basis probabilities",
))

그림 왼쪽은 격자 배치, 오른쪽은 채점기가 정한 실행 시점입니다.
작은 빨간 점은 noise가 들어가는 위치이며 기다리는 동안에도 나타납니다.
Histogram은 Z 기저 측정 확률입니다. **같은 확률분포여도 상대위상이 다르면 다른 state**이므로,
회로 비교에는 fidelity를 사용하세요.

### 2.3 튜토리얼 미션 2 — 같은 GHZ를 더 정확하게

### 먼저 적용해 볼 전략

1. **불필요한 gate 삭제:** 같은 qubit의 연속된 H 두 개는 이상적으로 항등 연산입니다.
   Noise가 있는 gate를 두 번 실행하면 오류 위치가 늘어납니다.
2. **병렬로 퍼뜨리기:** 이미 얽힌 qubit에서 양쪽으로 CNOT을 뻗으면, 서로 겹치지 않는 연산을
   동시에 할 수 있습니다. 줄 순서만 바꾸는 것으로는 부족하며 qubit 사이의 의존관계를 바꿔야 합니다.
   채점기는 독립적인 연산을 가능한 한 일찍 실행합니다.
3. **이동이 적은 위치 선택:** 자주 연산하는 qubit을 이웃하게 놓으세요. 일반적인 이동에는
   CNOT 세 개로 만든 SWAP 등이 필요합니다. 이번 미션은 0–1–2–3에 고정해 첫 두 전략에 집중합니다.

아래는 두 전략을 따로 적용한 예시입니다. 그림과 fidelity를 비교해 보세요.

In [ ]:
trimmed = QuantumCircuit(4)
trimmed.h(0)
trimmed.cx(0, 1)
trimmed.cx(1, 2)
trimmed.cx(2, 3)

parallel = QuantumCircuit(4)
parallel.h(1)
parallel.cx(1, 2)
parallel.cx(1, 0)
parallel.cx(2, 3)  # 직전 CNOT과 qubit이 겹치지 않아 같은 시점에 실행됩니다.

for label, circuit in [("H 두 개 삭제", trimmed), ("가운데서 양쪽으로", parallel)]:
    print(label)
    show_ghz(circuit)
    ghz_report(circuit)

### 직접 개선해 보기

`starter_ghz()`를 복사한 아래 회로를 수정하세요. 위 예시에서 시작해도 됩니다.

- Noise 없이 4-qubit GHZ를 만들어야 합니다.
- Qubit은 0, 1, 2, 3만 사용하고 이웃한 쌍에만 CNOT을 적용합니다.
- `h`, `cx`, `id`, `barrier`만 사용합니다. 측정·reset은 이번 미션에서 사용하지 않습니다.
- **p=0.01에서 시작 회로보다 fidelity를 0.005 이상 높이면 통과**합니다.

최적 회로를 찾을 필요는 없습니다. 개선 방법 하나를 적용하고 그 효과를 확인하는 미션입니다.

In [ ]:
def mission2_circuit():
    qc = starter_ghz()
    # TODO: 불필요한 gate를 지우거나 GHZ를 퍼뜨리는 구조를 바꿔 보세요.
    return qc


m2 = mission2_circuit()
show_ghz(m2)
ghz_report(m2)

In [ ]:
def check_mission2(circuit, participant):
    participant = participant.strip()
    if not participant or participant == "여기에 이름 또는 닉네임":
        print("PARTICIPANT에 이름 또는 닉네임을 입력해 주세요.")
        return None
    try:
        ghz_text(circuit)
        ideal = float(state_fidelity(Statevector.from_instruction(circuit), ghz_target(4)))
        if not np.isclose(ideal, 1.0, atol=1e-10, rtol=0):
            print("먼저 noise 없이 올바른 GHZ를 만드세요.")
            return None
        reference = float(state_fidelity(noisy_ghz(starter_ghz()), ghz_target(4)))
        fidelity = ghz_report(circuit)
    except ValueError as error:
        print(error)
        return None
    print(f"시작 회로 대비 fidelity 개선: {fidelity-reference:.6f}")
    if fidelity < reference + 0.005 - 1e-10:
        print("아직 개선 폭이 부족합니다. 회로를 수정하고 다시 실행하세요.")
        return None
    return mission_token(circuit, 2, participant)


check_mission2(m2, participant=PARTICIPANT)

미션을 수정했다면 회로 생성 셀과 확인 셀을 다시 실행하세요.
두 미션을 통과했다면 확인 화면을 저장해 두세요.

Gate 수와 깊이는 개선할 곳을 찾는 지표입니다. 최종적으로는 noise 모델에서 fidelity를 비교해야 합니다.
이 실습은 최종 오류정정 없이 GHZ를 평가합니다. 메인 문제는 목표가 7-qubit state로 바뀌며,
채점기가 제공하는 최종 정정 후의 infidelity를 평가합니다.

## Part 3. 선택 심화 — 측정으로 실행을 선택하고 보정하기

**이 파트는 선택 사항입니다. 내용을 몰라도 Part 4로 넘어가 메인 문제를 시작할 수 있습니다.**
예시 회로의 gate·배치·깊이를 개선하는 데 이 개념들이 꼭 필요한 것은 아닙니다.
측정을 이용한 검사나 실행 선택, Pauli 보정을 탐색하고 싶을 때 읽어 보세요.
추가 미션 없이 따라 실행하는 실습입니다.
앞에서는 오류가 덜 생기는 회로를 만들었습니다. 이번에는 **오류가 생긴 실행을 알아내서 버릴 수 있을까?**를 생각해 봅니다.

### 3.1 확인하고도 state를 남겨 둘 수 있을까요?

원하는 GHZ를 Z 기저로 측정하면 모든 qubit의 값이 같습니다.
한 qubit에 X 오류가 생기면 그 qubit만 다른 값이 됩니다.
그렇다면 모두 측정해서 같은 값인지 확인하면 될까요?

그러면 GHZ의 중첩이 사라집니다. 준비한 state를 나중에 사용하려면 **검사한 뒤에도 state가 남아 있어야** 합니다.
대신 qubit 0과 1에 대해 **각각의 값이 아니라 ‘둘이 같은가?’만** 묻겠습니다.
이것은 $Z_0Z_1$의 측정입니다.

| 결과 | 의미 | 저장하는 bit |
|---|---|---|
| +1 | 두 qubit의 Z 측정 값이 같음 | 0 |
| −1 | 두 qubit의 Z 측정 값이 다름 | 1 |

GHZ는 $Z_0Z_1$의 +1 고유상태입니다. 따라서 이상적인 검사에서는 항상 +1이 나오고 GHZ가 그대로 남습니다.
이처럼 원하는 state에 적용하면 state를 그대로 두는 Pauli operator를 **$\color{blue}{\text{stabilizer}}$**라고 부릅니다.

아래 회로는 보조 qubit 4에 패리티만 옮겨 측정합니다. Data qubit 0–3은 직접 측정하지 않습니다.
**이 파트의 작은 회로는 측정 원리를 설명하는 연결 제약 없는 예시**입니다.
메인 제출에는 격자 라우팅과 검사 회로 자체의 noise도 고려해야 합니다.

In [ ]:
def parity_demo(error_qubit=None):
    qc = QuantumCircuit(5, 1)
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.cx(2, 3)
    if error_qubit is not None:
        qc.x(error_qubit)  # 검사 전에 일부러 넣는 오류
    qc.cx(0, 4)
    qc.cx(1, 4)
    qc.measure(4, 0)
    return qc


display(parity_demo().draw("mpl"))

### 측정 결과별로 남는 state 확인하기

아래 도구는 마지막 보조 qubit의 측정 결과로 state를 나누고, 보조 qubit을 제외한 출력을 계산합니다.
기록이 0인 실행과 1인 실행의 확률, 각 실행의 GHZ fidelity를 볼 수 있습니다.
도구 내부는 그대로 두고 결과를 살펴보세요.

In [ ]:
from qiskit.quantum_info import partial_trace


def split_last_qubit(state):
    rho = DensityMatrix(state)
    dim = len(rho.data)
    n = int(np.log2(dim))
    branches = {}
    for m in (0, 1):
        mask = (np.arange(dim) >> (n - 1)) == m
        selected = rho.data * mask[:, None] * mask[None, :]
        probability = float(np.trace(selected).real)
        if probability > 1e-12:
            branches[m] = (probability, partial_trace(DensityMatrix(selected / probability), [n - 1]))
    return branches


for label, fault in [("오류 없음", None), ("qubit 0에 X", 0), ("qubit 2에 X", 2)]:
    circuit = parity_demo(fault)
    before_measure = circuit.remove_final_measurements(inplace=False)
    branches = split_last_qubit(Statevector.from_instruction(before_measure))
    print(label)
    for m, (probability, output) in branches.items():
        fidelity = state_fidelity(output, ghz_target(4))
        print(f"  bit={m}, 확률={probability:.3f}, 검사 후 GHZ fidelity={fidelity:.3f}")

정상 GHZ는 bit 0을 내고 그대로 남습니다. Qubit 0의 X 오류는 bit 1로 발견할 수 있지만,
qubit 2의 X 오류는 이 검사에서 보이지 않습니다.

**−1이 나왔다고 검사가 오류를 고쳐 주는 것은 아닙니다.** 오류의 흔적을 알려주므로 그 실행을 버릴 수 있는 것입니다.
반대로 **+1은 모든 오류가 없다는 보장이 아니라, 이 검사를 통과했다는 뜻**입니다.
일반적인 다른 state에서는 두 결과가 확률적으로 나올 수 있고, 측정으로 state가 바뀔 수도 있습니다.

### 3.2 버리고 나면 얼마나 좋아질까요?

이번에는 확률 0.15로 qubit 0에 X 오류가 생긴 상황을 만듭니다.
검사 전 fidelity와, bit 0인 실행만 남긴 후 fidelity를 비교합니다.
**여기서는 검사 원리를 분리해서 보기 위해 초기화와 gate는 이상적이고 X 오류 하나만 확률적으로 넣습니다.**
Part 2와 메인 문제에서는 검사에 쓰는 gate·측정에도 noise가 들어갑니다.

In [ ]:
prepare5 = QuantumCircuit(5)
prepare5.h(0)
prepare5.cx(0, 1)
prepare5.cx(1, 2)
prepare5.cx(2, 3)
rho5 = DensityMatrix.from_instruction(prepare5)
error_probability = 0.15
flip = Kraus([np.sqrt(1-error_probability) * np.eye(2),
              np.sqrt(error_probability) * Pauli("X").to_matrix()])
rho5 = rho5.evolve(flip, qargs=[0])
before_fidelity = state_fidelity(partial_trace(rho5, [4]), ghz_target(4))

parity_check = QuantumCircuit(5)
parity_check.cx(0, 4)
parity_check.cx(1, 4)
branches = split_last_qubit(rho5.evolve(parity_check))
acceptance, accepted_output = branches[0]
print(f"검사 전 fidelity: {before_fidelity:.3f}")
print(f"남긴 실행의 비율 (acceptance): {acceptance:.3f}")
print(f"남긴 실행의 fidelity: {state_fidelity(accepted_output, ghz_target(4)):.3f}")
print("버린 실행의 비율:", round(1-acceptance, 3))

### 3.3 DETECTOR와 POSTSELECT 선언

실행 일부를 버리고 남기는 것을 **post-selection**이라고 합니다.
메인 문제에서는 두 종류의 주석으로 조건을 지정합니다.

| 선언 | 무엇을 남길까요? |
|---|---|
| `// DETECTOR c[0]` | 정상 실행의 기준값과 같은 측정 결과 |
| `// POSTSELECT(0) c[0]` | bit가 0인 실행 |
| `// POSTSELECT(1) c[0]` | bit가 1인 실행 |

위 패리티 검사는 정상 GHZ에서 항상 0이므로 DETECTOR로 지정할 수 있습니다.
반면 **정상 실행에서도 원래 무작위인 결과 중 한쪽을 고르려면 POSTSELECT**를 사용합니다.
채점기는 DETECTOR의 기준값을 찾지만, POSTSELECT의 선택값은 참가자가 정합니다.
여러 bit를 적으면 그 bit들의 XOR를 사용하고, 여러 조건을 적으면 모두 만족해야 accept합니다.
DETECTOR는 POSTSELECT로 선택한 noiseless 실행들 안에서만 결정적이어도 됩니다.

다음 코드는 주석을 만드는 방법입니다. 위 GHZ 회로는 메인 목표 state 준비 회로가 아니며,
격자 조건도 적용하지 않았으므로 메인 문제에 제출하는 예시는 아닙니다.

In [ ]:
detector_example = tprep.submission_text(
    parity_demo(), detectors=[[('c', 0)]],
)
print(detector_example)

# 원래 무작위인 결과는 POSTSELECT로 선택합니다.
random_measure = QuantumCircuit(1, 1)
random_measure.h(0)
random_measure.measure(0, 0)
print(tprep.submission_text(random_measure, postselect=[(0, [('c', 0)])]))

무작위 보조 qubit의 bit 0만 선택한다고 출력이 좋아지지는 않습니다.
**선택하는 결과가 출력의 품질과 관련되어야** 유용합니다.
또 실행을 버릴수록 준비에 드는 자원이 늘어나므로 메인 문제에는 acceptance 하한이 있습니다.

### 3.4 다른 출력도 되돌릴 수 있다면 사용할 수 있을까요?

측정 결과가 다르다고 반드시 버릴 필요는 없습니다.
다음 두 state는 상대위상만 다르며, qubit 하나에 Z를 적용하면 서로 바뀝니다.

$$|\mathrm{GHZ}_{+}\rangle=\frac{|0000\rangle+|1111\rangle}{\sqrt2},\qquad
|\mathrm{GHZ}_{-}\rangle=\frac{|0000\rangle-|1111\rangle}{\sqrt2}.$$

**측정 기록으로 어느 쪽인지 안다면 두 결과를 모두 사용할 수 있습니다.**
다음 회로는 5-qubit GHZ를 만든 뒤 마지막 qubit을 X 기저로 측정합니다.
Qiskit에서는 H 다음 Z 측정으로 작성합니다. 결과를 bit $m$으로 기록하면:

| 기록 $m$ | 남은 qubit 0–3의 출력 | 목표 GHZ+로 돌아가는 보정 |
|---|---|---|
| 0 | GHZ+ | 없음 |
| 1 | GHZ− | qubit 0에 Z |

이 경우 bit 1을 버릴 수도 있지만, 알고 있는 Z 보정을 반영하면 모든 실행을 사용할 수 있습니다.

In [ ]:
frame_demo = QuantumCircuit(5, 1)
frame_demo.h(0)
for q in range(4):
    frame_demo.cx(q, q + 1)
frame_demo.h(4)
frame_demo.measure(4, 0)
display(frame_demo.draw("mpl"))

branches = split_last_qubit(Statevector.from_instruction(
    frame_demo.remove_final_measurements(inplace=False)
))
for m, (probability, output) in branches.items():
    correction = QuantumCircuit(4)
    if m == 1:
        correction.z(0)
    corrected = output.evolve(correction)
    print(f"bit={m}, 확률={probability:.3f}, "
          f"보정 전 F={state_fidelity(output, ghz_target(4)):.3f}, "
          f"보정 후 F={state_fidelity(corrected, ghz_target(4)):.3f}")

어떤 Pauli 보정이 필요한지 저장한 고전 정보를 **Pauli frame**이라고 부릅니다.
실제로 gate를 즉시 적용하는 대신 기록으로 관리하고, 이후 연산이나 결과 해석에 반영할 수 있습니다.
위 코드의 조건문은 분기별 state를 확인하기 위한 고전 계산입니다. 메인 제출에는 조건부 gate를 넣을 수 없습니다.

메인 문제에서는 $|T_L\rangle$뿐 아니라, **기록으로 구분할 수 있는**
$X_L|T_L\rangle$, $Y_L|T_L\rangle$, $Z_L|T_L\rangle$도 허용합니다.
여기서 $X_L=X^{\otimes7}$, $Z_L=Z^{\otimes7}$는 출력 일곱 qubit 모두에 X 또는 Z를 적용하는 연산입니다.
예를 들어 출력이 $X_L^m|T_L\rangle$이라면 $m=1$일 때 $X_L$을 한 번 더 적용하면 목표로 돌아옵니다.
기록을 모르거나, 실제 출력과 관계없는 기록을 선언하는 것으로는 올바른 보정이 되지 않습니다.

메인 회로의 기록 $m$이 `c[0]`이라면 다음 선언을 사용합니다.
**위 GHZ의 qubit 하나에 대한 Z와, 메인 문제의 logical Z는 다른 보정**입니다.
GHZ 실습 회로에 아래 logical FRAME을 그대로 붙이는 것은 아닙니다.

```text
// FRAME_X(0) c[0]
// FRAME_Z(0)
```

| 기능 | 하는 일 |
|---|---|
| DETECTOR / POSTSELECT | 어떤 실행을 남길지 정함 |
| FRAME | 남긴 출력에 어떤 logical Pauli 보정을 반영할지 정함 |

### 3.5 메인 목표에서는 무엇을 검사할까요?

메인 목표 state를 그대로 두는 stabilizer들은 다음과 같이 알려져 있습니다.
유도 과정을 알지 못해도 이 operator를 검사 회로 설계의 출발점으로 사용할 수 있습니다.

| Qubit 묶음 | 두 $\color{blue}{\text{stabilizer}}$ generator |
|---|---|
| $\{0,2,4,6\}$ | $X_0X_2X_4X_6$, $Z_0Z_2Z_4Z_6$ |
| $\{1,2,5,6\}$ | $X_1X_2X_5X_6$, $Z_1Z_2Z_5Z_6$ |
| $\{3,4,5,6\}$ | $X_3X_4X_5X_6$, $Z_3Z_4Z_5Z_6$ |

번호는 출력의 순서 $q_0,\ldots,q_6$이며 physical qubit 번호와 다를 수 있습니다.
**이 여섯 검사를 모두 통과해도 $|T_L\rangle$이라는 보장은 없습니다.**
다른 logical state도 같은 결과를 내므로 목표의 상대위상까지 맞아야 합니다.
검사 gate의 오류, 측정 오류, 기다리는 시간까지 고려해 실제 점수를 비교하세요.
자세한 제출 규칙은 `problem.md` 1.3·1.6·2.1절에 있습니다.

## Part 4. 메인 해커톤 — 원하는 state를 더 정확하게

Quantum computing에서는 알고리즘의 입력이나 연산에 필요한 state를 준비합니다.
메인 해커톤에서는 **7-qubit state 하나를 가능한 한 정확하게 만드는 회로**를 설계합니다.
튜토리얼 미션 2에서 본 상대위상이 더 많은 항에 들어 있습니다.

아래 식은 $|q_6q_5\dots q_0\rangle$ 순서입니다. 표시하지 않은 항의 amplitude는 모두 0입니다.

$$
\begin{aligned}
|\psi_{\mathrm{target}}\rangle={}&\frac14\bigl(
|0000000\rangle+|0011110\rangle+|0101101\rangle+|0110011\rangle\\
&\qquad+|1001011\rangle+|1010101\rangle+|1100110\rangle+|1111000\rangle\bigr)\\
&+\frac{e^{i\pi/4}}4\bigl(
|0000111\rangle+|0011001\rangle+|0101010\rangle+|0110100\rangle\\
&\qquad+|1001100\rangle+|1010010\rangle+|1100001\rangle+|1111111\rangle\bigr).
\end{aligned}
$$

16개 항의 확률은 각각 $1/16$입니다. 확률뿐 아니라 두 묶음 사이의 상대위상도 맞아야 합니다.
이 state의 이름은 $|T_L\rangle$입니다. 이름과 배경은 `problem.md`의 접힌 설명에서 볼 수 있습니다.

In [ ]:
# 목표를 검증용 Statevector로 적습니다. 회로를 만드는 코드는 아닙니다.
ZERO_TERMS = ["0000000", "0011110", "0101101", "0110011",
              "1001011", "1010101", "1100110", "1111000"]
ONE_TERMS = ["0000111", "0011001", "0101010", "0110100",
             "1001100", "1010010", "1100001", "1111111"]
amplitudes = np.zeros(128, dtype=complex)
for bits in ZERO_TERMS:
    amplitudes[int(bits, 2)] = 1 / 4
for bits in ONE_TERMS:
    amplitudes[int(bits, 2)] = np.exp(1j * np.pi / 4) / 4
main_target = Statevector(amplitudes)
print("정규화 확인:", main_target.is_valid())

### Amplitude에서 준비 회로 만들기

지금까지는 gate를 직접 골랐습니다. Qiskit은 원하는 amplitude를 입력받아 준비 회로를 만들 수도 있습니다.
먼저 4-qubit GHZ로 사용법을 봅시다. `prepare_state`를 분해하면 내부 gate가 나타납니다.

In [ ]:
automatic_ghz = QuantumCircuit(4)
automatic_ghz.prepare_state(ghz_target(4).data, range(4))
automatic_ghz = transpile(automatic_ghz, basis_gates=["u", "cx"], optimization_level=1)
display(automatic_ghz.draw("mpl"))
print("GHZ fidelity:", state_fidelity(Statevector.from_instruction(automatic_ghz), ghz_target(4)))

메인 목표도 같은 방법으로 준비할 수 있습니다.
다만 자동 생성 회로에는 일반적인 회전 gate인 `u` 등이 들어갑니다.
**이 회로는 state 준비를 이해하기 위한 예시이며, 메인 문제의 허용 gate 조건을 만족하는 제출은 아닙니다.**
자동 생성은 이상적인 state를 만들지만 gate 오류나 격자 이동 비용을 최소화한다고 보장하지 않습니다.

In [ ]:
automatic_main = QuantumCircuit(7)
automatic_main.prepare_state(main_target.data, range(7))
automatic_main = transpile(automatic_main, basis_gates=["u", "cx"], optimization_level=1)
print("자동 준비 회로의 gate 수:", dict(automatic_main.count_ops()))
display(automatic_main.draw("mpl", fold=70))
print("목표와의 fidelity:", state_fidelity(Statevector.from_instruction(automatic_main), main_target))

### 목표 state의 구조를 이용한 출발 회로

다음 예시는 일반 회전 gate 대신 **H, T, CNOT**으로 같은 목표를 만듭니다.
먼저 qubit 0에 H와 T를 적용해 magic state를 만들고, 나머지 연산으로 7-qubit에 encoding합니다.
처음부터 모든 CNOT의 역할을 해석할 필요는 없습니다.
앞에서처럼 이상적인 출력을 확인하고, 배치와 noise를 고려해 개선하는 출발점으로 쓰세요.

In [ ]:
def baseline_circuit():
    qc = QuantumCircuit(7)
    qc.h(0)
    qc.t(0)
    qc.h(1)
    qc.cx(0, 4)
    qc.cx(4, 0)
    qc.cx(0, 4)
    qc.cx(1, 0)
    qc.h(2)
    qc.cx(1, 5)
    qc.cx(5, 1)
    qc.cx(1, 5)
    qc.cx(2, 1)
    qc.cx(4, 5)
    qc.cx(5, 2)
    qc.cx(2, 4)
    qc.h(3)
    qc.cx(5, 6)
    qc.cx(4, 3)
    qc.cx(3, 5)
    qc.cx(5, 4)
    qc.cx(4, 6)
    qc.cx(6, 3)
    qc.cx(3, 6)
    qc.cx(6, 3)
    return qc

baseline = baseline_circuit()
display(baseline.draw("mpl", fold=70))
print("목표와의 fidelity:",
      state_fidelity(Statevector.from_instruction(baseline), main_target))

In [ ]:
chip = CouplingMap.from_grid(6, 6)
routed_main = transpile(baseline, coupling_map=chip,
                        basis_gates=["h", "t", "cx"],
                        optimization_level=1, seed_transpiler=1)
data_qubits = routed_main.layout.final_index_layout()[:7]
print("data_qubits =", data_qubits)
print("gate 수:", dict(routed_main.count_ops()))
display(routed_main.draw("mpl", idle_wires=False, fold=80))

전체 회로는 36-qubit으로 확장됩니다. **이 회로에 `Statevector`를 적용하지 마세요.**
36-qubit statevector는 일반적인 노트북의 메모리에 너무 큽니다. 메인 문제의 채점기 `tprep`를 사용합니다.
앞에서 7-qubit 회로로 확인한 fidelity는 **noise 없는 정확성**이며, 메인 점수와는 다릅니다.

메인에서는 다음을 요구합니다.

- Noise가 없을 때 목표 state를 정확히 준비할 것.
- 6×6 격자의 이웃한 qubit끼리 허용 gate를 사용할 것 (2-qubit gate는 `cx`만).
- 실행 중 측정으로 일부 결과를 버릴 수 있지만, noise가 있을 때도 절반 이상은 받아들일 것.
- **채점기가 제공하는 최종 정정 절차를 거친 뒤**, 받아들인 결과의 목표 state에 대한 infidelity를 최소화할 것.

최종 정정 절차와 정확한 noise·제출 규칙은 [`problem.md`](https://github.com/SNUQuantums/snu-qff2026-problem/blob/main/problem.md)에 있습니다.
회로를 간결하게 만들거나 qubit 배치를 바꾸는 시도부터 시작해도 좋습니다.

### 선택 실습: 메인 채점기로 점수 확인하기

앞에서 설치한 채점기로 메인 예시의 점수도 확인할 수 있습니다.
아래 셀의 `RUN_MAIN_JUDGE`를 `True`로 바꾸세요. 실행 시간이 걸릴 수 있습니다.
미션 1·2의 기념품 확인에는 필요하지 않습니다.


In [ ]:
RUN_MAIN_JUDGE = False

if RUN_MAIN_JUDGE:
    import tprep
    report = tprep.check(routed_main, data_qubits, shots=100_000, seed=0)
    print(report)
    submission = tprep.submission_text(routed_main)
    print("\n제출용 OpenQASM 2.0:\n", submission)
else:
    print("튜토리얼 완료! 메인 채점기를 설치했다면 RUN_MAIN_JUDGE = True로 바꾸세요.")

## 더 살펴보기

- [Qiskit QuantumCircuit API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit)
- [Statevector API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector)
- [state_fidelity와 quantum_info](https://quantum.cloud.ibm.com/docs/en/api/qiskit/quantum_info)
- [Bit 순서](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering)
- [TranspileLayout](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.TranspileLayout)
- [메인 해커톤 문제](https://github.com/SNUQuantums/snu-qff2026-problem/blob/main/problem.md)

AI에 질문할 때는 실행한 코드와 오류 메시지를 함께 주면 도움이 됩니다.
예를 들어 “이 Qiskit 회로의 출력 state를 단계별로 설명해 줘” 또는 “확률은 같은데 fidelity가 1이 아닌 이유를 설명해 줘”처럼 물어보세요.